#ZADANIE 1

In [0]:
from pyspark.sql.functions import explode, col, explode_outer

path = "/FileStore/tables/json/brzydki.json"
raw_df = spark.read.option("multiline", "true").json(path)

features_df = raw_df.select(explode(col("features")).alias("feature"))

flatten_df = features_df.select(
    col("feature.properties.featureId").alias("feature_id"),
    col("feature.properties.toid").alias("target_object"), 
    col("feature.properties.changeEventType").alias("change_event_type"),
    col("feature.properties.jobReference").alias("job_reference"),
    col("feature.properties.validFromTimestamp").alias("valid_from"),
    col("feature.geometry.type").alias("geometry_type"),
    col("feature.geometry.coordinates").alias("coordinates"),
    col("feature.properties.baseFormComponent.form").alias("base_form"),
    col("feature.properties.lifecycleStatusComponent.lifecycleStatus").alias("lifecycle_status"),
    col("feature.properties.baseFunctionComponents")[0]["function"].alias("base_function")
)

flatten_df.show(truncate=False)

+------------------------------------+--------------------+-----------------+-------------+--------------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+----------------+-----------------+
|feature_id                          |target_object       |change_event_type|job_reference|valid_from          |geometry_type|coordinates                                                                                                                                                |base_form         |lifecycle_status|base_function    |
+------------------------------------+--------------------+-----------------+-------------+--------------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+----------------+--------

In [0]:
features_df.printSchema() #to do sprawdzenia schematu 

root
 |-- feature: struct (nullable = true)
 |    |-- geometry: struct (nullable = true)
 |    |    |-- coordinates: array (nullable = true)
 |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |    |-- type: string (nullable = true)
 |    |-- properties: struct (nullable = true)
 |    |    |-- accessTopologyComponent: string (nullable = true)
 |    |    |-- administrativeExceptionComponent: string (nullable = true)
 |    |    |-- administrativeUnitComponent: string (nullable = true)
 |    |    |-- alternativeToids: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- anomalyComponent: string (nullable = true)
 |    |    |-- areaHeightComponent: string (nullable = true)
 |    |    |-- baseFormComponent: struct (nullable = true)
 |    |    |    |-- applicableContracts: array (nullable = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |

#ZADANIE 3

###1. Walidacja danych wejściowych
Cel: Zapewnienie, że dane wchodzące do pipeline spełniają określone wymagania i nie zawierają nieprawidłowych lub brakujących wartości.[](url)

In [0]:
def validate_input_data(df):
    # Sprawdzenie czy dane wejściowe zawierają wymagane kolumny
    required_columns = ["feature_id", "geometry_type", "coordinates"]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Brak wymaganych kolumn: {', '.join(missing_columns)}")
    
    # Walidacja typów danych (np. id musi być liczbą całkowitą, współrzędne - lista)
    if not df.filter(col("feature_id").isNotNull() & col("coordinates").isNotNull()).count() == df.count():
        raise ValueError("Brakujące dane w kolumnach 'feature_id' lub 'coordinates'")
    return df

###2. Zabezpieczenie przed pustymi lub błędnymi danymi
Cel: Upewnienie się, że dane w pipeline są kompletne. Może to obejmować usuwanie lub naprawianie brakujących danych w zbiorach danych.

In [0]:
def handle_missing_values(df):
    # Uzupełnianie brakujących wartości domyślnymi danymi (np. "Unknown" dla stringów, 0 dla liczb)
    df = df.fillna({
        'feature_id': 'Unknown',
        'geometry_type': 'Unknown',
        'coordinates': [0, 0]
    })
    
    # Alternatywnie możemy wyrzucać wiersze z brakującymi wartościami w krytycznych kolumnach
    df = df.dropna(subset=["feature_id", "geometry_type", "coordinates"])
    return df

###3. Logowanie i monitorowanie błędów
Cel: Zapewnienie śledzenia błędów i logowania istotnych informacji w czasie rzeczywistym, co pozwala na szybkie wykrycie problemów.

In [0]:
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

def log_pipeline_step(step_name):
    logging.info(f"Rozpoczęcie kroku pipeline: {step_name}")

def log_error(exception, step_name):
    logging.error(f"Błąd w kroku {step_name}: {exception}")

###4. Obsługa wyjątków w pipeline
Cel: Zapewnienie, że błędy w pipeline nie zatrzymają całego procesu, a pipeline będzie kontynuował przetwarzanie nawet w przypadku wystąpienia błędu w jednym z kroków.

In [0]:
def safely_process_step(df, step_function):
    try:
        df = step_function(df)
    except Exception as e:
        log_error(e, step_function.__name__)
        # W razie błędu zwrócenie danych wejściowych lub pustego DataFrame
        df = df.limit(0)
    return df

2025-05-13 11:08:21,258 - Received command c on object id p0


###5. Testowanie i walidacja wyników po każdej operacji
Cel: Upewnienie się, że po każdej operacji w pipeline dane są w odpowiednim formacie i nie zawierają nieoczekiwanych wartości.

In [0]:
def validate_output(df, expected_columns):
    # Sprawdzenie czy dataframe zawiera wszystkie wymagane kolumny
    missing_columns = [col for col in expected_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Brakujące kolumny w wyniku: {', '.join(missing_columns)}")
    
    # Testowanie, czy dane nie zawierają nieoczekiwanych wartości
    if df.filter(col("geometry_type") == "Invalid").count() > 0:
        raise ValueError("Znaleziono nieprawidłowy typ geometrii")
    return df

2025-05-13 11:08:34,084 - Received command c on object id p0
